# Gold — Sales Fact
One row per order line. Natural keys from the CRM are replaced by the surrogate keys of the dimensions, so this must run **after** `dim_customers` and `dim_products`.

→ `gold.fact_sales`

## Transformation (SQL)

In [ ]:
CATALOG = "workspace"

query = f"""
SELECT
    sd.order_number,
    pr.product_key,
    cu.customer_key,
    sd.order_date,
    sd.ship_date,
    sd.due_date,
    sd.sales_amount,
    sd.quantity,
    sd.price
FROM {CATALOG}.silver.crm_sales sd
LEFT JOIN {CATALOG}.gold.dim_products pr
    ON sd.product_number = pr.product_number
LEFT JOIN {CATALOG}.gold.dim_customers cu
    ON sd.customer_id = cu.customer_id
"""

df = spark.sql(query)

## Sanity check

In [ ]:
df.limit(10).display()

## Write gold table

In [ ]:
df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(f"{CATALOG}.gold.fact_sales")

In [ ]:
%sql
SELECT COUNT(*) AS rows FROM workspace.gold.fact_sales;